In [1]:
import xarray as xr
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from os.path import join
import os
from pathlib import Path

In [2]:
reference_forecast_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/baselines/mpas_forecasts"
baseline_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/baselines"

In [3]:
for make_persistence in [True, False]:
    save_forecast_dir = join(baseline_dir, "persistence_forecasts" if make_persistence else "climo_forecasts")
    os.makedirs(save_forecast_dir, exist_ok=True)
    
    if make_persistence:
        zarr_ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr", consolidated=False).drop_duplicates(
            dim="t"
        ).sortby(
            "t"
        ).transpose(
            "t", "channel", "latitude", "longitude")
    else:
        climo = (xr.open_dataset("/glade/derecho/scratch/dkimpara/goes_10km_train/baselines/climo/data_stats.nc")
                 .rename_vars(mean="BT_or_R")
                 .drop_vars("std")
                 .expand_dims("t")
        )
    
    dirs = [d for d in Path(reference_forecast_dir).iterdir() if d.is_dir()]
    
    for d in dirs:
        save_dir = join(save_forecast_dir, d.name)
        os.makedirs(save_dir, exist_ok=True)
    
        # the init time is the directory name
        ts = pd.Timestamp(d.name)
    
        if make_persistence:
            pred_ds = zarr_ds.sel(t=[ts], method="nearest")
        else:
            pred_ds = climo
    
        nc_filenames = [f.name for f in d.iterdir() if ".nc" in f.name]
    
        for fn in nc_filenames: 
            # save the "prediction" with forecast time
            # corresponding to reference forecast files
            
            pred_ds = pred_ds.copy()
            pred_ds["t"] = [pd.Timestamp(fn[:-3])]
            pred_ds.to_netcdf(join(save_dir, fn), mode="w")
    
        